# 📈 Notebook 3 – Demand Forecasting
**RetailPulse | Prophet + LSTM Ensemble | Target MAPE ≤ 12%**

This notebook covers:
1. Time-series decomposition & stationarity tests
2. Prophet baseline model
3. LSTM model with PyTorch Lightning
4. Ensemble blending
5. MLflow experiment tracking


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import mlflow
import os
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '../..')
plt.style.use('dark_background')
print('Libraries loaded ✅')

## 1. Load & Prepare Data

In [ ]:
df = pd.read_parquet('../../data/processed/retail_clean.parquet')
daily = df.groupby('Date')['TotalPrice'].sum().reset_index()
daily.columns = ['ds','y']
daily['ds'] = pd.to_datetime(daily['ds'])
daily = daily.sort_values('ds').reset_index(drop=True)

# Clip outliers
cap = daily['y'].quantile(0.99)
daily['y'] = daily['y'].clip(upper=cap)

print(f'Daily series: {len(daily)} days')
print(f'Date range: {daily["ds"].min().date()} → {daily["ds"].max().date()}')
print(f'Revenue range: £{daily["y"].min():.0f} – £{daily["y"].max():.0f}')
daily.head()

## 2. Time Series Decomposition

In [ ]:
daily_indexed = daily.set_index('ds')['y']
result = seasonal_decompose(daily_indexed, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(15, 10))
result.observed.plot(ax=axes[0],  color='#3b82f6'); axes[0].set_title('Observed')
result.trend.plot(ax=axes[1],     color='#f59e0b'); axes[1].set_title('Trend')
result.seasonal.plot(ax=axes[2],  color='#10b981'); axes[2].set_title('Seasonal')
result.resid.plot(ax=axes[3],     color='#ef4444'); axes[3].set_title('Residual')
plt.tight_layout()
plt.savefig('../../reports/forecasting_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Stationarity Test (ADF)

In [ ]:
adf_result = adfuller(daily['y'].dropna())
print('ADF Stationarity Test')
print(f'  ADF Statistic : {adf_result[0]:.4f}')
print(f'  p-value       : {adf_result[1]:.4f}')
print(f'  Critical (5%) : {adf_result[4]["5%"]:.4f}')
if adf_result[1] < 0.05:
    print('  ✅ Series is STATIONARY')
else:
    print('  ⚠️  Series is NON-STATIONARY – differencing needed')

## 4. Train / Test Split

In [ ]:
HORIZON = 30
split = daily['ds'].max() - pd.Timedelta(days=HORIZON)
train = daily[daily['ds'] <= split]
test  = daily[daily['ds'] > split]
print(f'Train: {len(train)} days | Test: {len(test)} days')

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(train['ds'], train['y'], color='#3b82f6', label='Train', linewidth=1.2)
ax.plot(test['ds'],  test['y'],  color='#10b981', label='Test',  linewidth=1.5)
ax.axvline(split, color='red', linestyle='--', label='Split')
ax.set_title('Train / Test Split')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Prophet Model

In [ ]:
os.environ['MLFLOW_TRACKING_URI'] = '../../mlruns'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_experiment('RetailPulse')

from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error

with mlflow.start_run(run_name='prophet_notebook'):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                daily_seasonality=False, changepoint_prior_scale=0.05)
    m.add_country_holidays(country_name='GB')
    m.fit(train)

    future = m.make_future_dataframe(periods=HORIZON)
    forecast = m.predict(future)
    test_forecast = forecast[forecast['ds'].isin(test['ds'])]

    mape_prophet = mean_absolute_percentage_error(test['y'].values, test_forecast['yhat'].values) * 100
    print(f'Prophet MAPE: {mape_prophet:.2f}%')
    mlflow.log_metric('prophet_mape', mape_prophet)

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
ax.plot(train['ds'], train['y'], color='#3b82f6', alpha=0.6, label='Train')
ax.plot(test['ds'],  test['y'],  color='#10b981', linewidth=2, label='Actual')
ax.plot(test_forecast['ds'], test_forecast['yhat'], color='#f59e0b', linewidth=2, linestyle='--', label='Prophet Forecast')
ax.fill_between(test_forecast['ds'], test_forecast['yhat_lower'], test_forecast['yhat_upper'], alpha=0.2, color='#f59e0b')
ax.axvline(split, color='red', linestyle='--', alpha=0.5)
ax.set_title(f'Prophet Forecast | MAPE: {mape_prophet:.2f}%', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig('../../reports/forecasting_prophet.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. MAPE Target Check

In [ ]:
TARGET_MAPE = 12.0
print(f'\n=== Model Performance Summary ===')
print(f'Prophet MAPE : {mape_prophet:.2f}% {"✅" if mape_prophet <= TARGET_MAPE else "❌"} (Target ≤ {TARGET_MAPE}%)')
print(f'\nNote: LSTM + Ensemble steps run in src/models/forecasting.py')
print(f'Full pipeline: python run_pipeline.py --step forecasting')